# 09 Validation

**Purpose:** Validate barcode classification and measurement performance against clinician labels and preprocessing QC metadata.

**Inputs:**
- `data/processed/labels/clinician/barcode_labels.csv`: clinician-provided reference labels.
- `data/processed/predictions/barcode_predictions.csv`: ResNet predictions.
- `data/processed/features/barcode_bscan_measurements.csv`: B-scan-level barcode measurements.
- `data/processed/features/barcode_volume_measurements.csv`: volume-level barcode measurements.
- `data/processed/qc/preprocessing_qc.csv`: preprocessing QC metadata.

**Main task:**
Evaluate whether the automated pipeline reliably detects and quantifies barcoding.

**Planned workflow:**
1. Load clinician labels, predictions, measurements, and QC.
2. Merge all outputs by patient ID, file name, and optionally B-scan index.
3. Evaluate classification:
   - sensitivity,
   - specificity,
   - PPV,
   - NPV,
   - ROC-AUC,
   - PR-AUC,
   - calibration.
4. Evaluate measurement stability:
   - robustness by scan quality,
   - robustness by ROI size,
   - robustness by B-scan count,
   - comparison across visits if available.
5. Review false positives and false negatives.
6. Save validation tables and figures.

**Code organization:**
- Long metric and validation functions go in `src/barcode/validation.py`.
- Plotting helpers can later go in `src/barcode/visualization.py`.
- The notebook should focus on loading outputs, calling validation functions, and reviewing summary tables/figures.

**Expected outputs:**
- `data/processed/validation/classification_metrics.csv`
- `data/processed/validation/calibration_metrics.csv`
- `data/processed/validation/error_analysis.csv`
- `data/processed/figures/validation/`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

LABEL_FILE = PROCESSED_DIR / "labels" / "clinician" / "barcode_labels.csv"
PREDICTION_FILE = PROCESSED_DIR / "predictions" / "barcode_volume_predictions.csv"

In [ ]:
from barcode.metrics import (
    run_validation_pipeline,
    merge_measurements_with_labels,
    summarize_measurements,
)

In [ ]:
if LABEL_FILE.exists() and PREDICTION_FILE.exists():
    print("Labels found:", LABEL_FILE)
    print("Predictions found:", PREDICTION_FILE)
else:
    print("Missing required validation inputs.")
    print("Labels exist:", LABEL_FILE.exists())
    print("Predictions exist:", PREDICTION_FILE.exists())

In [ ]:
if LABEL_FILE.exists() and PREDICTION_FILE.exists():
    outputs = run_validation_pipeline(
        processed_dir=PROCESSED_DIR,
        label_file=LABEL_FILE,
        prediction_file=PREDICTION_FILE,
        label_col="barcode_volume_status",
        prob_col="mean_barcode_prob",
        threshold=0.5,
        n_boot=1000,
    )

In [ ]:
if LABEL_FILE.exists() and PREDICTION_FILE.exists():
    display(outputs["metrics_df"])

In [ ]:
if LABEL_FILE.exists() and PREDICTION_FILE.exists():
    display(outputs["threshold_df"])

In [ ]:
if LABEL_FILE.exists() and PREDICTION_FILE.exists():
    display(outputs["bootstrap_df"])

In [ ]:
MEASUREMENT_FILE = PROCESSED_DIR / "features" / "barcode_volume_measurements.csv"

if LABEL_FILE.exists() and MEASUREMENT_FILE.exists():
    measure_df = merge_measurements_with_labels(
        processed_dir=PROCESSED_DIR,
        label_file=LABEL_FILE,
        measurement_file=MEASUREMENT_FILE,
    )

    measurement_summary_df = summarize_measurements(measure_df)

    display(measure_df.head())
    display(measurement_summary_df)
else:
    print("Measurement validation skipped.")
    print("Labels exist:", LABEL_FILE.exists())
    print("Measurements exist:", MEASUREMENT_FILE.exists())

In [ ]:
for path in [
    PROCESSED_DIR / "validation" / "classification_metrics.csv",
    PROCESSED_DIR / "validation" / "roc_curve.csv",
    PROCESSED_DIR / "validation" / "precision_recall_curve.csv",
    PROCESSED_DIR / "validation" / "threshold_search.csv",
    PROCESSED_DIR / "validation" / "bootstrap_metric_ci.csv",
]:
    print(path, "exists:", path.exists())